# 🚀 Pipeline DataOps — Telco Customer Churn
**Asignatura:** ITY1101 — Gestión de Datos para IA  
**Dataset:** Telco Customer Churn (7.043 clientes, 21 columnas)  
**Objetivo:** Predecir qué clientes abandonarán el servicio (Churn)

---
Este notebook ejecuta el pipeline DataOps completo y realiza un análisis exploratorio del dataset.

## 📦 1. Importaciones

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías importadas correctamente')

: 

## 📥 2. ETAPA 1 — Ingesta de Datos

In [ ]:
# Cargar el dataset original
df_raw = pd.read_csv('IA_Proyecto/data/telco_raw.csv')

print('=' * 50)
print('INGESTA COMPLETADA')
print('=' * 50)
print(f'Filas cargadas  : {len(df_raw)}')
print(f'Columnas        : {len(df_raw.columns)}')
print(f'Memoria usada   : {df_raw.memory_usage().sum() / 1024:.1f} KB')
df_raw.head()

In [ ]:
# Tipos de datos y valores faltantes
print('TIPOS DE DATOS Y VALORES FALTANTES')
print('-' * 40)
info = pd.DataFrame({
    'Tipo': df_raw.dtypes,
    'Nulos': df_raw.isnull().sum(),
    'Nulos %': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
print(info)

## 🧹 3. ETAPA 2 — Limpieza y Transformación

In [ ]:
df = df_raw.copy()

# Transformación 1: TotalCharges vacíos → imputar con MonthlyCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
nulos_antes = df['TotalCharges'].isna().sum()
df.loc[df['TotalCharges'].isna(), 'TotalCharges'] = df.loc[df['TotalCharges'].isna(), 'MonthlyCharges']
print(f'✅ T1: {nulos_antes} valores imputados en TotalCharges')

# Transformación 2: SeniorCitizen 0/1 → No/Yes
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
print('✅ T2: SeniorCitizen convertido a No/Yes')

# Transformación 3: Eliminar customerID
df = df.drop(columns=['customerID'])
print('✅ T3: customerID eliminado (anonimización - Ley 21.719)')

# Transformación 4: Eliminar duplicados
duplicados = df.duplicated().sum()
df = df.drop_duplicates()
print(f'✅ T4: {duplicados} filas duplicadas eliminadas')

print(f'\nShape final: {df.shape}')
print(f'Nulos residuales: {df.isnull().sum().sum()}')

## ✅ 4. ETAPA 3 — Validación

In [ ]:
errores = 0

# Validación estructural
print('── VALIDACIÓN ESTRUCTURAL')
nulos = df.isnull().sum().sum()
print(f'  Nulos residuales: {nulos} → {"✅ OK" if nulos == 0 else "❌ ERROR"}')

for col in ['MonthlyCharges', 'TotalCharges', 'tenure']:
    es_num = pd.api.types.is_numeric_dtype(df[col])
    print(f'  {col} numérico: {"✅ OK" if es_num else "❌ ERROR"}')
    if not es_num: errores += 1

# Validación semántica
print('\n── VALIDACIÓN SEMÁNTICA')
r1 = df[(df['PhoneService'] == 'No') & (df['MultipleLines'] != 'No phone service')]
print(f'  PhoneService/MultipleLines: {len(r1)} inconsistencias → {"✅ OK" if len(r1)==0 else "❌ ERROR"}')

r2 = df[(df['tenure'] < 0) | (df['tenure'] > 72)]
print(f'  tenure en rango [0,72]: {len(r2)} fuera de rango → {"✅ OK" if len(r2)==0 else "❌ ERROR"}')

print(f'\nTotal errores: {errores}')
print('✅ VALIDACIÓN COMPLETADA — datos listos para carga' if errores == 0 else '❌ VALIDACIÓN FALLIDA')

## 💾 5. ETAPA 4 — Carga

In [ ]:
import os
os.makedirs('IA_Proyecto/data', exist_ok=True)

ruta_salida = 'IA_Proyecto/data/telco_limpio.csv'
df.to_csv(ruta_salida, index=False)

# Verificar carga
df_verificacion = pd.read_csv(ruta_salida)
print('=' * 50)
print('CARGA COMPLETADA')
print('=' * 50)
print(f'Registros cargados : {len(df_verificacion)}')
print(f'Columnas cargadas  : {len(df_verificacion.columns)}')
print(f'Destino            : {ruta_salida}')
print(f'Tasa de error      : 0%')
print('✅ Carga exitosa')

## 📊 6. Análisis Exploratorio (EDA)

In [ ]:
print('ESTADÍSTICAS DESCRIPTIVAS — Variables Numéricas')
print('-' * 50)
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2))

In [ ]:
# Distribución de Churn
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: Distribución Churn
axes[0].bar(churn_counts.index, churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Churn', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Cantidad de clientes')
for i, (val, pct) in enumerate(zip(churn_counts.values, churn_pct.values)):
    axes[0].text(i, val + 50, f'{val}\n({pct:.1f}%)', ha='center', fontweight='bold')

# Gráfico 2: Tipo de contrato vs Churn
contrato_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
)
axes[1].bar(contrato_churn.index, contrato_churn.values, color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[1].set_title('Tasa de Churn por Tipo de Contrato', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tipo de Contrato')
axes[1].set_ylabel('Tasa de Churn (%)')
for i, val in enumerate(contrato_churn.values):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('IA_Proyecto/data/analisis_churn.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Gráficos generados')

In [ ]:
# Distribución de variables numéricas
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
colores = ['#3498db', '#9b59b6', '#e67e22']

for ax, col, color in zip(axes, cols, colores):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribución de {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Media: {df[col].mean():.1f}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Churn por método de pago
pago_churn = df.groupby('PaymentMethod')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(pago_churn.index, pago_churn.values, color='#e74c3c', alpha=0.8)
ax.set_title('Tasa de Churn por Método de Pago', fontsize=13, fontweight='bold')
ax.set_xlabel('Tasa de Churn (%)')
for bar, val in zip(bars, pago_churn.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 📋 7. Resumen Final del Pipeline

In [ ]:
print('=' * 55)
print('  RESUMEN PIPELINE DATAOPS — TELCO CUSTOMER CHURN')
print('=' * 55)
print(f'  Registros originales     : {len(df_raw)}')
print(f'  Valores imputados        : {nulos_antes} (TotalCharges)')
print(f'  Duplicados eliminados    : {duplicados}')
print(f'  Registros finales        : {len(df)}')
print(f'  Columnas finales         : {len(df.columns)}')
print(f'  Nulos residuales         : 0')
print(f'  Errores de validación    : 0')
print('-' * 55)
print(f'  Tasa de Churn            : {(df["Churn"]=="Yes").mean()*100:.1f}%')
print(f'  Clientes que se van      : {(df["Churn"]=="Yes").sum()}')
print(f'  Clientes que se quedan   : {(df["Churn"]=="No").sum()}')
print('=' * 55)
print('  ✅ Pipeline completado exitosamente')
print('=' * 55)